In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("../data/processed/listings_enriched.csv", low_memory=False)
df["occupancy_rate"] = 1 - (df["availability_365"] / 365)
df = df[df["price"].between(100, 10000)].copy()

print(f"Listings for recommendation: {len(df):,}")
print(df[["name","neighbourhood_cleansed","room_type","price"]].head(3))

Listings for recommendation: 22,801
                              name neighbourhood_cleansed        room_type  \
0  Nice room with superb city view            Ratchathewi  Entire Home/Apt   
3       Beautiful waterfront house             Don Mueang  Entire Home/Apt   
4  Condo with Chaopraya River View             Rat Burana     Private Room   

    price  
0  1595.0  
3  4188.0  
4  1450.0  


In [3]:
# --- Content-Based Recommendation System ---

# Build feature matrix for recommendations
rec_features = [
    "price", "accommodates", "bedrooms",
    "availability_365", "review_scores_rating",
    "neighbourhood_median_price", "occupancy_rate"
]

rec_df = df[["id", "name", "neighbourhood_cleansed", "room_type"] + rec_features].dropna().reset_index(drop=True)

# Scale numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rec_df[rec_features])

# Compute similarity matrix (sample 3000 for memory)
sample_df = rec_df.sample(3000, random_state=42).reset_index(drop=True)
X_sample = scaler.transform(sample_df[rec_features])
similarity_matrix = cosine_similarity(X_sample)

print(f"Similarity matrix: {similarity_matrix.shape}")

def recommend(listing_idx, top_k=5):
    """Recommend similar listings based on features"""
    sim_scores = list(enumerate(similarity_matrix[listing_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_k+1]  # exclude self
    
    print(f"\nQuery listing:")
    q = sample_df.iloc[listing_idx]
    print(f"  {q['name']} | {q['room_type']} | ฿{q['price']:,.0f} | {q['neighbourhood_cleansed']}")
    
    print(f"\nTop {top_k} recommendations:")
    for idx, score in sim_scores:
        r = sample_df.iloc[idx]
        print(f"  [{score:.3f}] {r['name']} | {r['room_type']} | ฿{r['price']:,.0f} | {r['neighbourhood_cleansed']}")

# Test recommendations
recommend(0)
recommend(100)
recommend(500)

Similarity matrix: (3000, 3000)

Query listing:
  3-Min to Ekkamai BTS | Fast Wi-Fi & Workspace | Entire Home/Apt | ฿1,693 | Vadhana

Top 5 recommendations:
  [0.998] Sukhumvit 5 Star Apt. 2PPL Skyview | Entire Home/Apt | ฿1,543 | Vadhana
  [0.997] Modern-Design Apt 45 sqm︱Ultra Fast WiFi︱@Ekamai | Entire Home/Apt | ฿1,487 | Vadhana
  [0.991] Sukhumvit Thonglor Condo High-End 2PPL Apt | Entire Home/Apt | ฿1,334 | Vadhana
  [0.991] Central Bangkok High-Rise | Walk to BTS & Shops | Entire Home/Apt | ฿1,570 | Vadhana
  [0.991] 曼谷市中心/像素大夏/帕蓬夜市/超大公寓/是隆沙吞 | Entire Home/Apt | ฿1,305 | Bang Rak

Query listing:
  NewCozy 1Bedroom OnnutBTS-Sukhumvit50 | Entire Home/Apt | ฿990 | Khlong Toei

Top 5 recommendations:
  [0.999] Bangkok so wander 1 bedroom 21 | Entire Home/Apt | ฿868 | Khlong Toei
  [0.998] Bright 1BR/Pool&Gym/W District/BTS Phra Khanong | Entire Home/Apt | ฿1,119 | Khlong Toei
  [0.997] Rooftop Bar, spacious stylish 1Bedder BTS/Discount | Entire Home/Apt | ฿1,160 | Khlong Toei
  [0.9

In [4]:
print("""
RECOMMENDATION SYSTEM EVALUATION
==================================
Method: Content-based filtering using cosine similarity
Features: Price, accommodates, bedrooms, availability, 
          rating, neighbourhood median price, occupancy rate
Sample: 3,000 listings (memory constraint)

Results:
- Recommendations are highly relevant (similarity 0.99+)
- System correctly identifies similar room types and price ranges
- Neighbourhood proximity naturally emerges from features

Cold-Start Problem & Solutions:
1. NEW LISTING (no reviews/bookings):
   - Use listing attributes only (current approach handles this)
   - Recommend based on host's existing listings if available
   - Default to neighbourhood median pricing as baseline

2. NEW USER (no preference history):
   - Ask for budget, room type, neighbourhood preference
   - Use popularity-based fallback (most reviewed listings)
   - Leverage demographic signals if available

3. SPARSE DATA neighbourhoods:
   - Less popular areas have fewer similar listings
   - Solution: expand similarity radius geographically

Collaborative Filtering Applicability:
- Would require user-listing interaction matrix (bookings/views)
- Inside Airbnb does not provide this data
- Could be approximated using review co-occurrence:
  users who reviewed listing A also reviewed listing B
- Matrix factorization (SVD) would surface latent preference factors

Production Enhancements:
1. Add text-based similarity using listing description TF-IDF
2. Include amenity vectors (WiFi, pool, parking flags)
3. Real-time personalization using booking history
4. A/B test recommendation strategies for CTR optimization
""")


RECOMMENDATION SYSTEM EVALUATION
Method: Content-based filtering using cosine similarity
Features: Price, accommodates, bedrooms, availability, 
          rating, neighbourhood median price, occupancy rate
Sample: 3,000 listings (memory constraint)

Results:
- Recommendations are highly relevant (similarity 0.99+)
- System correctly identifies similar room types and price ranges
- Neighbourhood proximity naturally emerges from features

Cold-Start Problem & Solutions:
1. NEW LISTING (no reviews/bookings):
   - Use listing attributes only (current approach handles this)
   - Recommend based on host's existing listings if available
   - Default to neighbourhood median pricing as baseline

2. NEW USER (no preference history):
   - Ask for budget, room type, neighbourhood preference
   - Use popularity-based fallback (most reviewed listings)
   - Leverage demographic signals if available

3. SPARSE DATA neighbourhoods:
   - Less popular areas have fewer similar listings
   - Solution: exp